# CIL Training — Equipo 25
# Conditional Imitation Learning (Codevilla, ICRA 2018)
# Referencia arquitectura: https://github.com/carla-simulator/imitation-learning
# MR4010 Proyecto Final — Tecnológico de Monterrey MNA

In [ ]:
import sys, os

IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    BASE = '/content/drive/MyDrive/MR4010_proyecto_final_2026'
else:
    BASE = os.path.normpath(os.path.join(os.path.dirname(os.path.abspath('__file__')), '..')) \
           if os.path.exists('__file__') else \
           '/Users/joelbecerril/MNA_WORKSPACE/NAVEGACION_AUTONOMA/Semana_9/ActFinal/MR4010_proyecto_final_2026'

# dataset_clean: filtrado de frames con cmd IZQ/DER pero steering incorrecto
# + sub-muestreo de CONTINUE recto (25% steer≈0, 10,883 filas)
CSV_PATH  = os.path.join(BASE, 'data', 'dataset_clean.csv')
MODEL_OUT = os.path.join(BASE, 'models', 'cil_model_equipo25.h5')
os.makedirs(os.path.join(BASE, 'models'), exist_ok=True)

print('BASE     :', BASE)
print('CSV      :', CSV_PATH)
print('MODEL_OUT:', MODEL_OUT)


In [ ]:
import csv, random
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, Model, Input
from PIL import Image, ImageEnhance
from collections import Counter

print('TensorFlow:', tf.__version__)
print('GPU disponible:', tf.config.list_physical_devices('GPU'))
print('CPU disponible:', tf.config.list_physical_devices('CPU'))

# Semilla reproducible
tf.random.set_seed(42)
random.seed(42)
np.random.seed(42)


In [ ]:
rows  = list(csv.DictReader(open(CSV_PATH)))
cmds  = Counter(int(r['nav_command']) for r in rows)
total = len(rows)
labels = {0:'CONTINUE', 1:'RECTO', 2:'IZQUIERDA', 3:'DERECHA'}

print(f'Total filas: {total:,}')
for c in range(4):
    n   = cmds[c]
    bar = '█' * int(n / total * 40)
    print(f'  {labels[c]:10s}: {n:6,} ({n/total*100:5.1f}%) {bar}')

zeros = sum(1 for r in rows if abs(float(r['steering_angle'])) < 0.01)
print(f'\nSteering ≈ 0 : {zeros:,} ({zeros/total*100:.1f}%)')
steers = [abs(float(r['steering_angle'])) for r in rows]
print(f'Steer máx    : {max(steers):.4f} rad')
print(f'Steer media  : {np.mean(steers):.4f} rad')


In [ ]:
IMG_W, IMG_H = 200, 88
N_COMMANDS   = 4
MAX_STEER    = 0.5
BATCH_SIZE   = 120
MEAN = np.array([0.485, 0.456, 0.406], dtype=np.float32)
STD  = np.array([0.229, 0.224, 0.225], dtype=np.float32)


# ── Augmentations ─────────────────────────────────────────────────────────────

def aug_brightness(img_arr):
    """Cambio de brillo ±30%."""
    factor = np.random.uniform(0.70, 1.30)
    return np.clip(img_arr * factor, 0.0, 1.0)

def aug_contrast(img_arr):
    """Cambio de contraste alrededor de la media."""
    factor = np.random.uniform(0.70, 1.30)
    mean   = img_arr.mean()
    return np.clip((img_arr - mean) * factor + mean, 0.0, 1.0)

def aug_shadow(img_arr):
    """
    Sombra triangular aleatoria (simula árboles, edificios).
    Oscurece una mitad de la imagen con un polígono aleatorio.
    """
    h, w, _ = img_arr.shape
    x1, x2  = np.random.randint(0, w, 2)
    mask     = np.zeros((h, w), dtype=np.float32)
    for row in range(h):
        col = int(x1 + (x2 - x1) * row / h)
        if np.random.rand() < 0.5:
            mask[row, :col] = 1
        else:
            mask[row, col:] = 1
    shadow  = np.random.uniform(0.45, 0.75)
    out     = img_arr.copy()
    out[mask == 1] *= shadow
    return np.clip(out, 0.0, 1.0)

def aug_noise(img_arr):
    """Ruido gaussiano leve."""
    noise = np.random.normal(0, 0.02, img_arr.shape).astype(np.float32)
    return np.clip(img_arr + noise, 0.0, 1.0)

def aug_zoom(img_arr):
    """Recorte central leve (zoom 5-15%) para simular variación de altura de cámara."""
    from PIL import Image as PILImage
    h, w, _ = img_arr.shape
    crop = np.random.uniform(0.05, 0.12)
    cy   = int(h * crop)
    cx   = int(w * crop)
    cropped = img_arr[cy:h-cy, cx:w-cx, :]
    out = np.array(PILImage.fromarray((cropped * 255).astype(np.uint8)).resize((w, h)),
                   dtype=np.float32) / 255.0
    return out

def apply_augmentation(img_arr):
    """Aplica augmentations aleatorias en secuencia."""
    if np.random.rand() < 0.6:
        img_arr = aug_brightness(img_arr)
    if np.random.rand() < 0.4:
        img_arr = aug_contrast(img_arr)
    if np.random.rand() < 0.5:
        img_arr = aug_shadow(img_arr)
    if np.random.rand() < 0.3:
        img_arr = aug_noise(img_arr)
    if np.random.rand() < 0.3:
        img_arr = aug_zoom(img_arr)
    return img_arr


def load_image(path, augment=False):
    from PIL import Image as PILImage
    img = PILImage.open(path).convert('RGB').resize((IMG_W, IMG_H))
    arr = np.array(img, dtype=np.float32) / 255.0
    if augment:
        arr = apply_augmentation(arr)
    arr = (arr - MEAN) / STD
    return arr


class CILSequence(keras.utils.Sequence):

    def __init__(self, rows, base_dir, augment=False):
        self.rows    = rows
        self.base    = base_dir
        self.augment = augment
        self.indices = list(range(len(rows)))
        if augment:
            random.shuffle(self.indices)

    def __len__(self):
        return int(np.ceil(len(self.rows) / BATCH_SIZE))

    def __getitem__(self, idx):
        batch_idx = self.indices[idx * BATCH_SIZE: (idx + 1) * BATCH_SIZE]
        imgs, spds, cmds, steers = [], [], [], []

        for i in batch_idx:
            row   = self.rows[i]
            steer = float(row['steering_angle'])
            cmd   = int(row['nav_command'])
            speed = float(row['speed_kmh']) / 30.0

            path = os.path.join(self.base, row['image_path'])
            flip = self.augment and random.random() < 0.5

            arr = load_image(path, augment=self.augment)
            if flip:
                arr   = arr[:, ::-1, :].copy()
                steer = -steer
                cmd   = 3 if cmd == 2 else (2 if cmd == 3 else cmd)

            steer_norm = float(np.clip(steer / MAX_STEER, -1.0, 1.0))
            imgs.append(arr)
            spds.append([speed])
            cmds.append(cmd)
            steers.append([steer_norm])

        return (
            {'image':   np.array(imgs,   dtype=np.float32),
             'speed':   np.array(spds,   dtype=np.float32),
             'command': np.array(cmds,   dtype=np.int32)},
            np.array(steers, dtype=np.float32)
        )

    def on_epoch_end(self):
        if self.augment:
            random.shuffle(self.indices)


by_cmd = {c: [r for r in rows if int(r['nav_command']) == c] for c in range(4)}
train_rows, val_rows = [], []
for c, rws in by_cmd.items():
    random.shuffle(rws)
    cut = int(len(rws) * 0.85)
    train_rows += rws[:cut]
    val_rows   += rws[cut:]
random.shuffle(train_rows)

train_gen = CILSequence(train_rows, BASE, augment=True)
val_gen   = CILSequence(val_rows,   BASE, augment=False)

print(f'Train: {len(train_rows):,} muestras ({len(train_gen)} batches)')
print(f'Val  : {len(val_rows):,} muestras ({len(val_gen)} batches)')
print()
print('Augmentations activas:')
print('  ✓ Flip horizontal + inversión steering + swap IZQ↔DER (p=0.5)')
print('  ✓ Brillo aleatorio ±30%          (p=0.6)')
print('  ✓ Contraste aleatorio ±30%       (p=0.4)')
print('  ✓ Sombra triangular aleatoria    (p=0.5)  ← nueva')
print('  ✓ Ruido gaussiano σ=0.02         (p=0.3)  ← nueva')
print('  ✓ Zoom aleatorio 5-12%           (p=0.3)  ← nueva')


In [ ]:
def build_cil_model(n_commands=4, img_h=88, img_w=200):
    """
    Arquitectura Codevilla 2018 (ICRA) implementada en Keras.
    Ref: github.com/carla-simulator/imitation-learning

    Entradas:
      image   : (H, W, 3) imagen de cámara normalizada
      speed   : (1,)      velocidad normalizada [0,1]
      command : ()        entero 0-3 (nav command)

    Salida:
      steering_norm : (1,) en [-1, 1]  →  multiplicar por MAX_STEER para rad
    """
    # ── Entradas ──────────────────────────────────────────────────────────────
    img_in = Input(shape=(img_h, img_w, 3), name='image')
    spd_in = Input(shape=(1,),              name='speed')
    cmd_in = Input(shape=(),  dtype='int32',name='command')

    # ── CNN backbone (8 capas conv, Codevilla) ────────────────────────────────
    # Bloque 1
    x = layers.Conv2D(32, 5, strides=2, padding='valid')(img_in)
    x = layers.BatchNormalization()(x)
    x = layers.Dropout(0.2)(x)
    x = layers.ReLU()(x)

    x = layers.Conv2D(32, 3, strides=1, padding='valid')(x)
    x = layers.BatchNormalization()(x)
    x = layers.Dropout(0.2)(x)
    x = layers.ReLU()(x)

    # Bloque 2
    x = layers.Conv2D(64, 3, strides=2, padding='valid')(x)
    x = layers.BatchNormalization()(x)
    x = layers.Dropout(0.2)(x)
    x = layers.ReLU()(x)

    x = layers.Conv2D(64, 3, strides=1, padding='valid')(x)
    x = layers.BatchNormalization()(x)
    x = layers.Dropout(0.2)(x)
    x = layers.ReLU()(x)

    # Bloque 3
    x = layers.Conv2D(128, 3, strides=2, padding='valid')(x)
    x = layers.BatchNormalization()(x)
    x = layers.Dropout(0.2)(x)
    x = layers.ReLU()(x)

    x = layers.Conv2D(128, 3, strides=1, padding='valid')(x)
    x = layers.BatchNormalization()(x)
    x = layers.Dropout(0.2)(x)
    x = layers.ReLU()(x)

    # Bloque 4
    x = layers.Conv2D(256, 3, strides=1, padding='valid')(x)
    x = layers.BatchNormalization()(x)
    x = layers.Dropout(0.2)(x)
    x = layers.ReLU()(x)

    x = layers.Conv2D(256, 3, strides=1, padding='valid')(x)
    x = layers.BatchNormalization()(x)
    x = layers.Dropout(0.2)(x)
    x = layers.ReLU()(x)

    # ── FC imagen ─────────────────────────────────────────────────────────────
    x = layers.Flatten()(x)
    x = layers.Dense(512, activation='relu')(x)
    x = layers.Dropout(0.2)(x)
    x = layers.Dense(512, activation='relu')(x)
    x = layers.Dropout(0.2)(x)

    # ── Rama de velocidad ─────────────────────────────────────────────────────
    s = layers.Dense(128, activation='relu')(spd_in)
    s = layers.Dense(128, activation='relu')(s)

    # ── Concatenar + FC conjunto ──────────────────────────────────────────────
    joined = layers.Concatenate()([x, s])
    joined = layers.Dense(512, activation='relu')(joined)

    # ── 4 ramas de comando (una por acción) ───────────────────────────────────
    branch_outs = []
    for i in range(n_commands):
        b = layers.Dense(256, activation='relu')(joined)
        b = layers.Dropout(0.5)(b)
        b = layers.Dense(256, activation='relu')(b)
        b = layers.Dense(1,   activation='tanh', name=f'branch_{i}')(b)
        branch_outs.append(b)                  # cada b: (batch, 1)

    # ── Selección de rama por comando (one-hot mask) ──────────────────────────
    # Concatenar todas las ramas: (batch, n_commands)
    all_branches = layers.Concatenate(name='all_branches')(branch_outs)

    # One-hot del comando para seleccionar la rama activa
    cmd_onehot = layers.Lambda(
        lambda c: tf.one_hot(c, n_commands), name='cmd_onehot'
    )(cmd_in)

    # Producto elemento a elemento + suma → steering de la rama seleccionada
    selected = layers.Multiply()([all_branches, cmd_onehot])
    output   = layers.Lambda(
        lambda t: tf.reduce_sum(t, axis=1, keepdims=True), name='steering'
    )(selected)

    model = Model(inputs=[img_in, spd_in, cmd_in], outputs=output, name='CIL_Codevilla')
    return model


model = build_cil_model(N_COMMANDS, IMG_H, IMG_W)
model.summary()
print(f'\nParámetros totales: {model.count_params():,}')


In [ ]:
EPOCHS = 40
LR     = 1e-4
TOL_RAD = 0.05   # tolerancia para métrica de accuracy (±0.05 rad ≈ ±2.9°)

# Loss ponderada: curvas (|steer_norm| > 0.2) pesan 5× más
def weighted_mse(y_true, y_pred):
    weight = 1.0 + 4.0 * tf.cast(tf.abs(y_true) > 0.2, tf.float32)
    return tf.reduce_mean(weight * tf.square(y_pred - y_true))

# Métrica: fracción de predicciones dentro de ±TOL_RAD (en rad)
def steer_accuracy(y_true, y_pred):
    err_rad = tf.abs(y_pred - y_true) * MAX_STEER
    return tf.reduce_mean(tf.cast(err_rad < TOL_RAD, tf.float32))

model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=LR),
    loss=weighted_mse,
    metrics=[steer_accuracy, 'mae']
)

callbacks = [
    keras.callbacks.ModelCheckpoint(
        MODEL_OUT, monitor='val_loss', save_best_only=True,
        verbose=1, mode='min'
    ),
    keras.callbacks.ReduceLROnPlateau(
        monitor='val_loss', factor=0.5, patience=7,
        min_lr=1e-6, verbose=1
    ),
    keras.callbacks.EarlyStopping(
        monitor='val_loss', patience=12, restore_best_weights=True, verbose=1
    ),
]

history = model.fit(
    train_gen,
    validation_data=val_gen,
    epochs=EPOCHS,
    callbacks=callbacks,
    verbose=1,
)


In [ ]:
import matplotlib.pyplot as plt

hist = history.history
epochs_ran = range(1, len(hist['loss']) + 1)

fig, axes = plt.subplots(1, 3, figsize=(16, 4))

axes[0].plot(epochs_ran, hist['loss'],     label='Train loss')
axes[0].plot(epochs_ran, hist['val_loss'], label='Val loss')
axes[0].set_title('Weighted MSE Loss')
axes[0].set_xlabel('Epoch'); axes[0].legend()

axes[1].plot(epochs_ran, [v * MAX_STEER for v in hist['mae']],     label='Train MAE')
axes[1].plot(epochs_ran, [v * MAX_STEER for v in hist['val_mae']], label='Val MAE')
axes[1].set_title('MAE (rad)')
axes[1].set_xlabel('Epoch'); axes[1].legend()

axes[2].plot(epochs_ran, hist['steer_accuracy'],     label='Train Acc')
axes[2].plot(epochs_ran, hist['val_steer_accuracy'], label='Val Acc')
axes[2].set_title(f'Accuracy (|err| < {TOL_RAD} rad)')
axes[2].set_xlabel('Epoch'); axes[2].legend()

plt.tight_layout()
plt.savefig(os.path.join(BASE, 'models', 'learning_curve.png'), dpi=120)
plt.show()

best_epoch = int(np.argmin(hist['val_loss'])) + 1
print(f"Mejor epoch    : {best_epoch}/{len(hist['loss'])}")
print(f"Mejor val_loss : {min(hist['val_loss']):.5f}")
print(f"Val MAE        : {hist['val_mae'][best_epoch-1]*MAX_STEER:.4f} rad")
print(f"Val Accuracy   : {hist['val_steer_accuracy'][best_epoch-1]*100:.1f}%")
print(f"Modelo guardado: {MODEL_OUT}")


In [ ]:
# Verificación del modelo guardado e inferencia de prueba
model_loaded = keras.models.load_model(
    MODEL_OUT,
    custom_objects={'weighted_mse': weighted_mse, 'steer_accuracy': steer_accuracy}
)
print(f"Modelo cargado: {MODEL_OUT}")
print(f"Entradas : {[i.name for i in model_loaded.inputs]}")
print(f"Salida   : {model_loaded.output.name}  shape={model_loaded.output.shape}")

# Inferencia con imagen negra (sanity check: salida no debe ser NaN)
dummy_img = np.zeros((1, IMG_H, IMG_W, 3), dtype=np.float32)
dummy_spd = np.array([[0.5]], dtype=np.float32)
label_map = {0:'CONTINUE', 1:'RECTO', 2:'IZQUIERDA', 3:'DERECHA'}

print('\nInferencia con imagen negra (debe ser ≠ NaN):')
for cmd in range(4):
    dummy_cmd = np.array([cmd], dtype=np.int32)
    pred_norm = model_loaded.predict(
        {'image': dummy_img, 'speed': dummy_spd, 'command': dummy_cmd},
        verbose=0
    )[0, 0]
    steer_rad = float(pred_norm) * MAX_STEER
    print(f"  CMD {cmd} ({label_map[cmd]:10s}): {steer_rad:+.4f} rad  (norm={pred_norm:+.4f})")

print('\nModelo Keras listo para autonomous_cil.py')
